# Decision Trees : IRIS Dataset

Le jeu de données Iris contient 150 fleurs réparties en trois espèces. Les quatre variables décrivent la longueur et la largeur des sépales et des pétales.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
iris = load_iris()

X = iris.data    # The inputs
y = iris.target  # The wanted output

df = pd.DataFrame(X,columns=iris.feature_names)
df['Label']=y
df['Species']=df['Label'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})
df = df.drop(['Label'], axis=1)

In [ ]:
df.head()   # The first five elements

In [ ]:
df.tail()  # the last five elements

In [ ]:
df.describe()   # statistical and general information about the data

In [ ]:
axes = pd.plotting.scatter_matrix(
    df.drop(columns="Species"),
    c=y,
    cmap="viridis",
    figsize=(11, 11),
    diagonal="hist",
    alpha=0.75,
)
plt.suptitle("Relations entre les variables du jeu Iris", y=0.92)
plt.show()

In [ ]:
# Modèle entraîné sur toutes les observations pour visualiser les règles apprises.
model_tree = DecisionTreeClassifier(random_state=42)

In [ ]:
# Training the model
model_tree.fit(X, y)

In [ ]:
text_representation = export_text(model_tree, feature_names = iris['feature_names'])
print(text_representation)

In [ ]:
fig = plt.figure(figsize=(25,20))
_   = plot_tree(model_tree, 
                feature_names=iris['feature_names'],  
                   class_names=['setosa', 'versicolor','virginica'],
                   filled=True)

In [ ]:
# Séparation stratifiée : 80 % pour l'entraînement et 20 % pour le test.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

rf_classifier = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_classifier.fit(X_train, y_train)

rf_predictions = rf_classifier.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_predictions)
print(f"Précision de la forêt aléatoire : {rf_accuracy:.2%}")

In [ ]:
# Évaluation équitable de l'arbre de décision sur le même jeu de test.
tree_classifier = DecisionTreeClassifier(random_state=42)
tree_classifier.fit(X_train, y_train)
tree_predictions = tree_classifier.predict(X_test)
tree_accuracy = accuracy_score(y_test, tree_predictions)

results = pd.DataFrame(
    {
        "Modèle": ["Arbre de décision", "Forêt aléatoire"],
        "Accuracy": [tree_accuracy, rf_accuracy],
    }
).sort_values("Accuracy", ascending=False, ignore_index=True)
display(results.style.format({"Accuracy": "{:.2%}"}))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    tree_predictions,
    display_labels=iris.target_names,
    cmap="Blues",
    ax=axes[0],
    colorbar=False,
)
axes[0].set_title("Arbre de décision")
ConfusionMatrixDisplay.from_predictions(
    y_test,
    rf_predictions,
    display_labels=iris.target_names,
    cmap="Greens",
    ax=axes[1],
    colorbar=False,
)
axes[1].set_title("Forêt aléatoire")
plt.tight_layout()
plt.show()